# Importing Libraries

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline

# Loading the Document

In [2]:
pdf_file = "Resume.pdf"
loader = PyPDFLoader(pdf_file)
documents = loader.load()

# Text Chunking

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)

# Embedding Generation

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding Model Loaded")

/tmp/ipykernel_29986/180835020.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Model Loaded


# Vector Database

In [5]:
db = FAISS.from_documents(
    chunks,
    embeddings
)
print("FAISS Vector Database Created")

FAISS Vector Database Created


# Language Model

In [8]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Language Model Loaded")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Language Model Loaded


In [10]:
while True:

    query = input("Ask a question (type exit to stop): ")

    if query.lower() == "exit":
        break

    results = db.similarity_search(query, k=2)

    context = "\n".join(
        [doc.page_content for doc in results]
    )

    prompt = f"""
    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("\nAnswer:")
    print(answer)

Ask a question (type exit to stop): in which company she is doing internship?

Answer:
Celebal Technologies
Ask a question (type exit to stop): what is the duration of the internship?

Answer:
8-week
Ask a question (type exit to stop): what is her current cgpa?

Answer:
8.3
Ask a question (type exit to stop): exit
